In [ ]:
# ============================================================
# DAY 09 - ARABIC OCR V3
# Spatial reconstruction from OCR bounding boxes
#
# IMPORTANT:
# This does NOT clean the old OCR.
# It goes directly back to the PDF.
#
# Main improvements:
#   1. Render PDF at 3x
#   2. OCR the high-resolution page
#   3. Extract text + confidence + coordinates
#   4. Reconstruct lines from coordinates
#   5. Sort Arabic lines top -> bottom
#   6. Sort words right -> left
#   7. Automatically insert spaces between OCR boxes
#   8. Preserve raw OCR boxes
#
# No package installation here.
# ============================================================

import os
import json
import re
import numpy as np
import cv2
import pymupdf

# ============================================================
# PADDLE ENVIRONMENT
# ============================================================

os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"

import torch
import paddle
import paddleocr
import paddlex

from paddleocr import PaddleOCR


# ============================================================
# PATHS
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"

PROJECT_DIR = os.path.join(
    BASE_DIR,
    "day09_rag"
)

PDF_PATH = os.path.join(
    BASE_DIR,
    "الجريدة-الرسمية-دولة-ليبيا-ديوان-مجلس-النواب-العدد-الثاني.pdf"
)

OCR_DIR = os.path.join(
    PROJECT_DIR,
    "01_ocr"
)

OUTPUT_JSON = os.path.join(
    OCR_DIR,
    "arabic_ocr_spatial.json"
)

OUTPUT_TXT = os.path.join(
    OCR_DIR,
    "arabic_ocr_spatial.txt"
)

os.makedirs(OCR_DIR, exist_ok=True)


# ============================================================
# SETTINGS
# ============================================================

SCALE = 3.0

MIN_CONFIDENCE = 0.25

# How close two boxes need to be vertically
# to belong to the same text line.
LINE_Y_TOLERANCE = 0.55


# ============================================================
# LOAD OCR
# ============================================================

print("Loading PaddleOCR...")

ocr = PaddleOCR(
    lang="ar",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

print("OCR loaded.")


# ============================================================
# HELPERS
# ============================================================

def polygon_to_bbox(poly):
    """
    Convert polygon points to:
        x1, y1, x2, y2
    """

    arr = np.asarray(poly, dtype=float)

    if arr.ndim == 1:
        arr = arr.reshape(-1, 2)

    x1 = float(np.min(arr[:, 0]))
    y1 = float(np.min(arr[:, 1]))
    x2 = float(np.max(arr[:, 0]))
    y2 = float(np.max(arr[:, 1]))

    return x1, y1, x2, y2


def get_box_from_result(data, index):

    # Preferred field
    if "rec_polys" in data:

        boxes = data["rec_polys"]

        if index < len(boxes):
            try:
                return polygon_to_bbox(boxes[index])
            except:
                pass

    # Alternative field
    if "rec_boxes" in data:

        boxes = data["rec_boxes"]

        if index < len(boxes):

            try:

                box = np.asarray(
                    boxes[index],
                    dtype=float
                ).flatten()

                if len(box) >= 4:

                    return (
                        float(box[0]),
                        float(box[1]),
                        float(box[2]),
                        float(box[3])
                    )

            except:
                pass

    return None


def extract_result(res):

    data = res.json

    if isinstance(data, str):
        data = json.loads(data)

    if "res" in data:
        data = data["res"]

    texts = data.get(
        "rec_texts",
        []
    )

    scores = data.get(
        "rec_scores",
        []
    )

    elements = []

    for i, text in enumerate(texts):

        text = str(text).strip()

        if not text:
            continue

        confidence = None

        if i < len(scores):

            try:
                confidence = float(
                    scores[i]
                )
            except:
                confidence = None

        box = get_box_from_result(
            data,
            i
        )

        if box is None:
            continue

        x1, y1, x2, y2 = box

        width = max(
            1.0,
            x2 - x1
        )

        height = max(
            1.0,
            y2 - y1
        )

        center_x = (
            x1 + x2
        ) / 2

        center_y = (
            y1 + y2
        ) / 2

        elements.append({
            "text": text,
            "confidence": confidence,
            "box": [
                x1,
                y1,
                x2,
                y2
            ],
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "center_x": center_x,
            "center_y": center_y,
            "width": width,
            "height": height
        })

    return elements


# ============================================================
# GROUP BOXES INTO LINES
# ============================================================

def group_into_lines(elements):

    if not elements:
        return []

    # Sort approximately top -> bottom
    elements = sorted(
        elements,
        key=lambda e: (
            e["center_y"],
            -e["center_x"]
        )
    )

    heights = [
        e["height"]
        for e in elements
        if e["height"] > 0
    ]

    if heights:

        median_height = float(
            np.median(heights)
        )

    else:

        median_height = 20.0

    tolerance = (
        median_height *
        LINE_Y_TOLERANCE
    )

    lines = []

    for element in elements:

        best_line = None
        best_distance = None

        for line in lines:

            distance = abs(
                element["center_y"]
                -
                line["center_y"]
            )

            if distance <= tolerance:

                if (
                    best_distance is None
                    or
                    distance < best_distance
                ):

                    best_distance = distance
                    best_line = line

        if best_line is None:

            lines.append({
                "center_y": element["center_y"],
                "elements": [element]
            })

        else:

            best_line["elements"].append(
                element
            )

            # Update line center
            best_line["center_y"] = np.mean(
                [
                    e["center_y"]
                    for e in best_line["elements"]
                ]
            )

    # Sort lines top -> bottom
    lines.sort(
        key=lambda line: line["center_y"]
    )

    # Sort words RIGHT -> LEFT
    for line in lines:

        line["elements"].sort(
            key=lambda e: e["center_x"],
            reverse=True
        )

    return lines


# ============================================================
# RECONSTRUCT TEXT FROM A LINE
# ============================================================

def reconstruct_line(line):

    elements = line["elements"]

    if not elements:
        return ""

    words = []

    for element in elements:

        text = element["text"].strip()

        if text:
            words.append(text)

    return " ".join(words)


# ============================================================
# REMOVE EXTREME OCR GARBAGE
# ============================================================

def basic_text_cleanup(text):

    if not text:
        return ""

    # Invisible characters
    text = re.sub(
        r"[\u200b\u200c\u200d\ufeff]",
        "",
        text
    )

    # Normalize spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# PROCESS PDF
# ============================================================

print()
print("=" * 70)
print("STARTING SPATIAL OCR")
print("=" * 70)

print("PDF:", PDF_PATH)
print("Render scale:", SCALE)

pdf = pymupdf.open(
    PDF_PATH
)

all_pages = []

total_elements = 0


# ============================================================
# PAGE LOOP
# ============================================================

for page_index in range(len(pdf)):

    page_number = page_index + 1

    print()
    print(
        f"Processing page "
        f"{page_number}/{len(pdf)}..."
    )

    page = pdf[page_index]

    # --------------------------------------------------------
    # Render at 3x
    # --------------------------------------------------------

    pix = page.get_pixmap(
        matrix=pymupdf.Matrix(
            SCALE,
            SCALE
        ),
        alpha=False
    )

    img = np.frombuffer(
        pix.samples,
        dtype=np.uint8
    )

    img = img.reshape(
        pix.height,
        pix.width,
        pix.n
    )

    if pix.n == 4:

        img = img[:, :, :3]

    # --------------------------------------------------------
    # RGB image
    # --------------------------------------------------------

    rgb = img

    # --------------------------------------------------------
    # Mild preprocessing
    #
    # We intentionally do NOT use aggressive thresholding.
    # It can destroy Arabic dots and diacritics.
    # --------------------------------------------------------

    gray = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2GRAY
    )

    gray = cv2.GaussianBlur(
        gray,
        (3, 3),
        0
    )

    processed = cv2.cvtColor(
        gray,
        cv2.COLOR_GRAY2RGB
    )

    # --------------------------------------------------------
    # OCR
    # --------------------------------------------------------

    results = ocr.predict(
        processed,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False
    )

    # --------------------------------------------------------
    # Extract all OCR boxes
    # --------------------------------------------------------

    page_elements = []

    for res in results:

        elements = extract_result(
            res
        )

        for element in elements:

            confidence = element[
                "confidence"
            ]

            if (
                confidence is not None
                and
                confidence < MIN_CONFIDENCE
            ):
                continue

            page_elements.append(
                element
            )

    # --------------------------------------------------------
    # Group spatially into lines
    # --------------------------------------------------------

    lines = group_into_lines(
        page_elements
    )

    # --------------------------------------------------------
    # Reconstruct page
    # --------------------------------------------------------

    reconstructed_lines = []

    for line in lines:

        text = reconstruct_line(
            line
        )

        text = basic_text_cleanup(
            text
        )

        if text:

            reconstructed_lines.append(
                text
            )

    page_text = "\n".join(
        reconstructed_lines
    )

    total_elements += len(
        page_elements
    )

    # --------------------------------------------------------
    # Save complete spatial information
    # --------------------------------------------------------

    page_data = {
        "page": page_number,

        "width": pix.width,
        "height": pix.height,

        "scale": SCALE,

        "text": page_text,

        "line_count": len(
            reconstructed_lines
        ),

        "element_count": len(
            page_elements
        ),

        "elements": page_elements,

        "lines": [
            {
                "text": reconstruct_line(
                    line
                ),
                "center_y": line[
                    "center_y"
                ],
                "elements": [
                    {
                        "text": e["text"],
                        "confidence": e[
                            "confidence"
                        ],
                        "box": e["box"],
                        "center_x": e[
                            "center_x"
                        ],
                        "center_y": e[
                            "center_y"
                        ]
                    }
                    for e in line["elements"]
                ]
            }
            for line in lines
        ]
    }

    all_pages.append(
        page_data
    )

    print(
        "  OCR boxes:",
        len(page_elements)
    )

    print(
        "  Lines:",
        len(reconstructed_lines)
    )


pdf.close()


# ============================================================
# SAVE JSON
# ============================================================

output_data = {

    "source_pdf": PDF_PATH,

    "ocr_version": "v3_spatial",

    "render_scale": SCALE,

    "preprocessing": [
        "grayscale",
        "GaussianBlur_3x3"
    ],

    "reconstruction": {
        "method": "bounding_box_spatial_order",
        "vertical_order": "top_to_bottom",
        "horizontal_order": "right_to_left",
        "line_tolerance": LINE_Y_TOLERANCE
    },

    "pages": len(
        all_pages
    ),

    "total_text_elements": total_elements,

    "pages_data": all_pages
}


with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_data,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# SAVE TXT
# ============================================================

with open(
    OUTPUT_TXT,
    "w",
    encoding="utf-8"
) as f:

    for page in all_pages:

        f.write("\n")
        f.write("=" * 80)
        f.write("\n")
        f.write(
            f"الصفحة {page['page']}"
        )
        f.write("\n")
        f.write("=" * 80)
        f.write("\n")

        f.write(
            page["text"]
        )

        f.write("\n")


# ============================================================
# FINISHED
# ============================================================

print()
print("=" * 70)
print("SPATIAL OCR FINISHED")
print("=" * 70)

print(
    "Pages:",
    len(all_pages)
)

print(
    "Total OCR boxes:",
    total_elements
)

print()
print("JSON:")
print(OUTPUT_JSON)

print()
print("TXT:")
print(OUTPUT_TXT)

print()
print("=" * 70)
print("FIRST PAGES PREVIEW")
print("=" * 70)

for page in all_pages[:3]:

    print()
    print(
        f"--- PAGE {page['page']} ---"
    )

    for line in page["text"].splitlines()[:30]:

        print(line)

In [ ]:
import os
import json
import re
from collections import Counter
from functools import lru_cache

# ============================================================
# PATHS
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"
PROJECT_DIR = os.path.join(BASE_DIR, "day09_rag")

INPUT_JSON = os.path.join(
    PROJECT_DIR,
    "01_ocr",
    "arabic_ocr_v2.json"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "02_cleaned_ocr"
)

OUTPUT_JSON = os.path.join(
    OUTPUT_DIR,
    "arabic_ocr_cleaned.json"
)

OUTPUT_TXT = os.path.join(
    OUTPUT_DIR,
    "arabic_ocr_cleaned.txt"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# LOAD OCR
# ============================================================

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

pages = data["pages_data"]

print("Loaded OCR")
print("Pages:", len(pages))

# ============================================================
# ARABIC NORMALIZATION
# ============================================================

def normalize_arabic(text):

    if not text:
        return ""

    text = str(text)

    # Remove invisible characters
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)

    # Arabic presentation forms
    text = re.sub(r"[\uFE70-\uFEFF]", "", text)

    # Normalize Alef
    text = re.sub(r"[إأآٱ]", "ا", text)

    # Normalize Yeh / Alef Maqsura
    text = text.replace("ى", "ي")

    # Normalize Teh Marbuta only for comparison
    # We DON'T change the actual displayed text here.

    # Remove tatweel
    text = text.replace("ـ", "")

    # Normalize punctuation spacing
    text = text.replace("،", " ، ")
    text = text.replace("؛", " ؛ ")
    text = text.replace(":", " : ")
    text = text.replace("(", " ( ")
    text = text.replace(")", " ) ")

    # Remove repeated spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def comparison_form(word):

    word = normalize_arabic(word)

    word = word.replace("ة", "ه")

    # Remove diacritics
    word = re.sub(r"[\u064B-\u065F\u0670]", "", word)

    return word


# ============================================================
# BASIC OCR GARBAGE
# ============================================================

GARBAGE_EXACT = {
    "S",
    "s",
    "w",
    "W",
    "=",
    "-",
    "_",
    "*",
    "ESEN",
    "لي",
    "مجس",
    "النو",
    "الليب",
}

def is_garbage(text):

    t = text.strip()

    if not t:
        return True

    if t in GARBAGE_EXACT:
        return True

    # Latin garbage with no Arabic
    if re.fullmatch(r"[A-Za-z]+", t):
        return True

    # Random symbols
    if re.fullmatch(r"[\W_]+", t, flags=re.UNICODE):
        return True

    return False


# ============================================================
# EXTRACT ALL TOKENS
# ============================================================

all_tokens = []

for page in pages:

    for element in page.get("elements", []):

        text = element.get("text", "")

        if not text:
            continue

        text = normalize_arabic(text)

        tokens = re.findall(
            r"[0-9٠-٩]+|[\u0600-\u06FF]+",
            text
        )

        for token in tokens:

            if len(token) >= 2:

                c = comparison_form(token)

                if c:
                    all_tokens.append(c)


# ============================================================
# BUILD DOCUMENT VOCABULARY
# ============================================================

freq = Counter(all_tokens)

# Words that appear in the document.
#
# This is important:
# If OCR produced:
#
#     مجلسالنواب
#
# but elsewhere we have:
#
#     مجلس
#     النواب
#
# the program can discover the correct split automatically.

vocabulary = set()

for word, count in freq.items():

    if len(word) >= 2:
        vocabulary.add(word)


# Add important Arabic grammatical/legal words.
# These are generic linguistic units, NOT corrections
# of specific OCR mistakes.

BASE_VOCAB = """
مجلس
النواب
الدولة
الحكومة
الوطنية
الليبية
القانون
قانون
قرار
قرارات
الدستوري
الدستور
الإعلان
المؤقت
الصادر
المادة
المواد
الأولى
الثانية
الثالثة
الرابعة
الخامسة
السادسة
السابعة
الثامنة
التاسعة
العاشرة
رئيس
رئاسة
الرئيس
نائب
نائبا
وزير
وزراء
وزارة
المحكمة
العليا
القضاء
النظام
الداخلي
انتخاب
انتخابات
المرحلة
الانتقالية
اعتماد
تعديل
تعديلات
إضافة
إلغاء
أحكام
حكم
بشأن
شأن
في
على
عن
من
إلى
و
او
أو
بعد
قبل
وعلى
والى
وبعد
وبشأن
بموجب
بناء
بناءً
صدر
يصدر
يعمل
يلغى
ينشر
الجريدة
الرسمية
مدينة
طبرق
بنغازي
تاريخ
بتاريخ
الموافق
يناير
فبراير
مارس
أبريل
مايو
يونيو
يوليو
أغسطس
سبتمبر
أكتوبر
نوفمبر
ديسمبر
محرم
صفر
ربيع
جمادى
رجب
شعبان
رمضان
شوال
القعدة
الحجة
الاجتماعي
الاجتماعية
العمل
العمال
الخدمة
المدنية
الضمان
الاستثمار
الاستثمارات
الاستثمارية
المصرف
مصرف
ليبيا
المركزي
المفوضية
الوطنية
العليا
الانتخابات
التشريعية
التنفيذية
السلطة
الرئاسية
البرلمانية
العضوية
الأعضاء
الاستقالة
إقالة
تكليف
الثقة
سحب
اسقاط
إسقاط
تشكيل
تشكيلة
وزارية
المعاشات
التقاعد
التأمين
العسكريين
الشركات
العامة
الخاصة
الدولة
النفطية
الصحة
التربية
التعليم
الثقافة
العدل
الداخلية
الخارجية
الدفاع
المواصلات
النقل
الاقتصاد
المالية
الزراعة
الصناعة
المعادن
السياحة
الرياضة
الشباب
الهجرة
المهجرين
المفقودين
المرأة
العمل
التأهيل
الكهرباء
الطاقات
المتجددة
الاتصالات
المعلومات
الاشغال
البحرية
الثروة
الموارد
المائية
الطيران
المحلية
الشؤون
الاجتماعية
""".split()

for word in BASE_VOCAB:
    vocabulary.add(comparison_form(word))


# ============================================================
# CHARACTER CONFUSIONS
# ============================================================

# These are ONLY used for comparison/fuzzy matching.
# They do not blindly replace characters in the output.

CONFUSION_GROUPS = [
    ("ا", "أ", "إ", "آ"),
    ("ه", "ة"),
    ("ي", "ى"),
    ("ت", "ة"),
]


def char_equivalent(a, b):

    if a == b:
        return True

    for group in CONFUSION_GROUPS:
        if a in group and b in group:
            return True

    return False


# ============================================================
# EDIT DISTANCE
# ============================================================

def edit_distance(a, b, limit=3):

    if a == b:
        return 0

    if abs(len(a) - len(b)) > limit:
        return limit + 1

    previous = list(range(len(b) + 1))

    for i, ca in enumerate(a, start=1):

        current = [i]

        min_row = i

        for j, cb in enumerate(b, start=1):

            cost = 0 if char_equivalent(ca, cb) else 1

            value = min(
                current[j - 1] + 1,
                previous[j] + 1,
                previous[j - 1] + cost
            )

            current.append(value)

            min_row = min(min_row, value)

        if min_row > limit:
            return limit + 1

        previous = current

    return previous[-1]


# ============================================================
# INDEX VOCABULARY BY LENGTH
# ============================================================

vocab_by_length = {}

for word in vocabulary:

    length = len(word)

    if length not in vocab_by_length:
        vocab_by_length[length] = []

    vocab_by_length[length].append(word)


# ============================================================
# FUZZY CORRECTION
# ============================================================

@lru_cache(maxsize=50000)
def fuzzy_correct(word):

    cword = comparison_form(word)

    if not cword:
        return word

    if cword in vocabulary:
        return word

    length = len(cword)

    # Very short words are dangerous to correct.
    if length <= 2:
        return word

    candidates = []

    # Search nearby word lengths.
    for L in range(
        max(2, length - 2),
        length + 3
    ):

        for candidate in vocab_by_length.get(L, []):

            # Do not compare extremely rare nonsense against
            # everything unless length is close.
            d = edit_distance(
                cword,
                candidate,
                limit=2
            )

            if d <= 2:

                frequency = freq.get(candidate, 0)

                candidates.append(
                    (d, -frequency, candidate)
                )

    if not candidates:
        return word

    candidates.sort()

    best_d, _, best = candidates[0]

    # Require stronger evidence for longer corrections.
    if best_d == 1:
        return best

    if best_d == 2 and length >= 6:
        return best

    return word


# ============================================================
# WORD SEGMENTATION
# ============================================================

@lru_cache(maxsize=50000)
def segment_word(word):

    word = comparison_form(word)

    if len(word) < 5:
        return [word]

    if word in vocabulary:
        return [word]

    n = len(word)

    # DP:
    # cost = number of unknown pieces + preference for
    # frequent words.

    dp = [None] * (n + 1)
    dp[0] = (0, [])

    for i in range(n):

        if dp[i] is None:
            continue

        current_cost, current_words = dp[i]

        # Try all possible word lengths.
        for j in range(i + 2, min(n, i + 18) + 1):

            piece = word[i:j]

            if piece in vocabulary:

                frequency = freq.get(piece, 1)

                # Frequent words are cheaper.
                cost = 1.0 / (1.0 + frequency)

                new_cost = current_cost + cost

                if dp[j] is None or new_cost < dp[j][0]:

                    dp[j] = (
                        new_cost,
                        current_words + [piece]
                    )

    if dp[n] is not None:

        result = dp[n][1]

        # Only accept segmentation into >=2 words.
        if len(result) >= 2:

            reconstructed = "".join(result)

            if reconstructed == word:

                return result

    return [word]


# ============================================================
# AUTOMATIC SPACING
# ============================================================

def repair_joined_words(text):

    # Process Arabic-looking tokens.
    parts = re.split(r"(\s+)", text)

    output = []

    for part in parts:

        if not part or part.isspace():
            output.append(part)
            continue

        # Ignore numbers / punctuation.
        if not re.fullmatch(r"[\u0600-\u06FF]+", part):
            output.append(part)
            continue

        segmented = segment_word(part)

        if len(segmented) > 1:

            output.append(" ".join(segmented))

        else:

            output.append(part)

    return "".join(output)


# ============================================================
# GENERIC OCR SPACING REPAIR
# ============================================================

def spacing_repair(text):

    # Arabic letters stuck to numbers.
    text = re.sub(
        r"([\u0600-\u06FF])([0-9٠-٩])",
        r"\1 \2",
        text
    )

    text = re.sub(
        r"([0-9٠-٩])([\u0600-\u06FF])",
        r"\1 \2",
        text
    )

    # Arabic punctuation.
    text = re.sub(
        r"\s+([،؛:])",
        r"\1",
        text
    )

    text = re.sub(
        r"([،؛:])(?=[\u0600-\u06FF])",
        r"\1 ",
        text
    )

    # Parentheses.
    text = re.sub(
        r"\(\s+",
        "(",
        text
    )

    text = re.sub(
        r"\s+\)",
        ")",
        text
    )

    # Multiple spaces.
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


# ============================================================
# CONTEXTUAL OCR CORRECTIONS
#
# These are not individual page corrections.
# They target common OCR corruption patterns.
# ============================================================

PATTERN_FIXES = [

    # Arabic OCR often loses spaces after prefixes.
    (r"\bوعلى(?=[\u0600-\u06FF])", "وعلى "),
    (r"\bوالى(?=[\u0600-\u06FF])", "والى "),
    (r"\bوعن(?=[\u0600-\u06FF])", "وعن "),
    (r"\bوبعد(?=[\u0600-\u06FF])", "وبعد "),
    (r"\bوبشأن(?=[\u0600-\u06FF])", "وبشأن "),

    # Common legal phrases.
    (r"فيشان", "في شأن"),
    (r"بشان(?=[\u0600-\u06FF])", "بشأن "),
    (r"بشأن(?=[\u0600-\u06FF])", "بشأن "),
    (r"لسنة(?=[\u0600-\u06FF])", "لسنة "),
    (r"للقانون", "للقانون"),

    # Common joined legal expressions.
    (r"الإعلانالدستوري", "الإعلان الدستوري"),
    (r"مجلسالنواب", "مجلس النواب"),
    (r"مجلسالدولة", "مجلس الدولة"),
    (r"رئيسالحكومة", "رئيس الحكومة"),
    (r"رئيسمجلس", "رئيس مجلس"),
    (r"المحكمةالعليا", "المحكمة العليا"),
    (r"نظامالقضاء", "نظام القضاء"),
    (r"النظامالداخلي", "النظام الداخلي"),
    (r"الضمانالاجتماعي", "الضمان الاجتماعي"),
    (r"علاقاتالعمل", "علاقات العمل"),
    (r"الجريدةالرسمية", "الجريدة الرسمية"),
    (r"مصرفليبيا", "مصرف ليبيا"),
    (r"المرحلةالانتقالية", "المرحلة الانتقالية"),
    (r"السلطةالتشريعية", "السلطة التشريعية"),
    (r"السلطةالتنفيذية", "السلطة التنفيذية"),
    (r"المفوضيةالوطنية", "المفوضية الوطنية"),
    (r"الانتخاباتالعامة", "الانتخابات العامة"),

    # Legal document markers.
    (r"المادةالأولى", "المادة الأولى"),
    (r"المادةالثانية", "المادة الثانية"),
    (r"المادةالثالثة", "المادة الثالثة"),
    (r"مادةمكرر", "مادة مكرر"),

    # Names/titles frequently joined by OCR.
    (r"السيدعلي", "السيد علي"),
    (r"السيدعبد", "السيد عبد"),
    (r"السيدمحمد", "السيد محمد"),
    (r"السيدخالد", "السيد خالد"),
    (r"السيدصالح", "السيد صالح"),
    (r"السيدفتحي", "السيد فتحي"),
    (r"السيدفائز", "السيد فائز"),
    (r"السيدزياد", "السيد زياد"),
]


def pattern_repair(text):

    for pattern, replacement in PATTERN_FIXES:
        text = re.sub(
            pattern,
            replacement,
            text,
            flags=re.UNICODE
        )

    return text


# ============================================================
# HIGH-CONFIDENCE OCR WORD CORRECTIONS
#
# Automatically learned from words appearing elsewhere.
# ============================================================

def correct_tokens(text):

    def replace_token(match):

        token = match.group(0)

        if len(token) <= 3:
            return token

        corrected = fuzzy_correct(token)

        return corrected

    return re.sub(
        r"[\u0600-\u06FF]+",
        replace_token,
        text
    )


# ============================================================
# CLEAN ONE LINE
# ============================================================

def clean_line(text):

    if not text:
        return ""

    # Basic cleanup
    text = normalize_arabic(text)

    # Remove obvious garbage.
    if is_garbage(text):
        return ""

    # Pattern repair
    text = pattern_repair(text)

    # Joined-word detection
    text = repair_joined_words(text)

    # Fuzzy correction
    text = correct_tokens(text)

    # Pattern repair again because fuzzy correction
    # can expose new boundaries.
    text = pattern_repair(text)

    # Spacing
    text = spacing_repair(text)

    # Remove accidental duplicated spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# PROCESS PAGES
# ============================================================

cleaned_pages = []

total_before = 0
total_after = 0

for page in pages:

    page_number = page["page"]

    cleaned_elements = []

    for element in page.get("elements", []):

        raw = element.get("text", "")

        if not raw:
            continue

        total_before += 1

        cleaned = clean_line(raw)

        if not cleaned:
            continue

        total_after += 1

        cleaned_elements.append({
            "text": cleaned,
            "raw_text": raw,
            "confidence": element.get("confidence"),
            "box": element.get("box")
        })

    page_text = "\n".join(
        e["text"]
        for e in cleaned_elements
    )

    cleaned_pages.append({
        "page": page_number,
        "text": page_text,
        "elements": cleaned_elements
    })


# ============================================================
# REMOVE EMPTY / GARBAGE PAGES
# ONLY FOR TEXT OUTPUT.
# PAGE STRUCTURE REMAINS IN JSON.
# ============================================================

# ============================================================
# SAVE JSON
# ============================================================

cleaned_data = {
    "source_pdf": data.get("source_pdf"),
    "ocr_version": data.get("ocr_version"),
    "cleaning_version": "v5_automatic",
    "method": [
        "Arabic normalization",
        "automatic vocabulary construction",
        "automatic joined-word segmentation",
        "document-wide fuzzy correction",
        "OCR spacing repair",
        "legal phrase boundary repair",
        "garbage removal"
    ],
    "pages": len(cleaned_pages),
    "text_elements_before": total_before,
    "text_elements_after": total_after,
    "pages_data": cleaned_pages
}

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cleaned_data,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# SAVE TXT
# ============================================================

with open(
    OUTPUT_TXT,
    "w",
    encoding="utf-8"
) as f:

    for page in cleaned_pages:

        f.write("\n")
        f.write("=" * 80)
        f.write("\n")
        f.write(f"الصفحة {page['page']}")
        f.write("\n")
        f.write("=" * 80)
        f.write("\n")

        if page["text"].strip():

            f.write(page["text"])

        f.write("\n")


# ============================================================
# REPORT
# ============================================================

print()
print("=" * 70)
print("AUTOMATIC OCR CLEANING COMPLETE")
print("=" * 70)
print()
print("Input:")
print(INPUT_JSON)
print()
print("Output JSON:")
print(OUTPUT_JSON)
print()
print("Output TXT:")
print(OUTPUT_TXT)
print()
print("Pages:", len(cleaned_pages))
print("Elements before:", total_before)
print("Elements after :", total_after)
print("Vocabulary size:", len(vocabulary))
print()
print("=" * 70)

In [ ]:
# ============================================================
# DAY 09 - FINAL LEGAL RAG CHUNKER
# ============================================================
#
# Goals:
#   1. Preserve 100% of source text
#   2. Legal section -> article -> meaningful topic
#   3. Avoid tiny 5-20 word chunks
#   4. Avoid giant 400+ word chunks
#   5. No arbitrary 350-word chunking
#   6. No random merging
#   7. Automatic keywords
#   8. Full coverage verification
# ============================================================

import os
import re
import json
import math
from collections import Counter


# ============================================================
# PATHS
# ============================================================

INPUT_FILE = (
    r"C:\Users\PC-LAB1\Desktop\llm\day09_rag"
    r"\02_cleaned_ocr\arabic_ocr_cleaned.json"
)

OUTPUT_DIR = (
    r"C:\Users\PC-LAB1\Desktop\llm\day09_rag"
    r"\03_chunks"
)

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "chunks.json"
)

PREVIEW_FILE = os.path.join(
    OUTPUT_DIR,
    "chunks_preview.txt"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


pages = data["pages_data"]


# ============================================================
# NORMALIZATION FOR ANALYSIS ONLY
# ============================================================

def normalize(text):

    if not text:
        return ""

    text = str(text)

    text = re.sub(
        r"[\u0617-\u061A\u064B-\u065F\u0670]",
        "",
        text
    )

    replacements = {
        "أ": "ا",
        "إ": "ا",
        "آ": "ا",
        "ٱ": "ا",
        "ى": "ي",
        "ة": "ه",
        "ؤ": "و",
        "ئ": "ي"
    }

    for old, new in replacements.items():
        text = text.replace(
            old,
            new
        )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def words(text):

    return re.findall(
        r"[\u0621-\u063A\u0641-\u064A]+",
        normalize(text)
    )


# ============================================================
# STOPWORDS
# ============================================================

STOPWORDS = {
    "من", "في", "على", "الى", "الي", "عن",
    "مع", "هذا", "هذه", "ذلك", "تلك",
    "التي", "الذي", "الذين", "كما",
    "كان", "كانت", "يكون", "تكون",
    "هو", "هي", "هم", "هن",
    "و", "او", "أو", "ثم",
    "ان", "أن", "إن", "لا", "لم", "لن",
    "ما", "مما", "بما", "به", "بها",
    "له", "لها", "لهم", "منه", "منها",
    "وقد", "قد", "تم", "يتم",
    "بعد", "قبل", "عند", "حتى",
    "كل", "أي", "اي", "بعض",
    "بين", "حيث", "وذلك",
    "عليه", "عليها", "عليهم",
    "وهو", "وهي"
}


GENERIC = {
    "رقم",
    "رق",
    "رقملسنه",
    "رقلسنه",
    "رقسنه",
    "سنه",
    "لسنه",
    "الماده",
    "المادة",
    "ماده",
    "بشان",
    "بشأن",
    "صدر",
    "مجلس",
    "النواب",
    "السيد",
    "الساده",
    "السادة",
    "وعلي",
    "وعلى",
    "علي",
    "على",
    "القانون"
}


# ============================================================
# OCR UNITS
# ============================================================

units = []

for page in pages:

    page_number = page.get(
        "page"
    )

    elements = page.get(
        "elements",
        []
    )

    if elements:

        for element in elements:

            text = element.get(
                "text",
                ""
            ).strip()

            if text:

                units.append({
                    "page": page_number,
                    "text": text
                })

    else:

        text = page.get(
            "text",
            ""
        ).strip()

        if text:

            units.append({
                "page": page_number,
                "text": text
            })


# ============================================================
# DETECTORS
# ============================================================

def is_constitutional(text):

    return bool(
        re.match(
            r"^التعديل\s*الدستوري",
            normalize(text)
        )
    )


def is_law(text):

    return bool(
        re.match(
            r"^قانون\b",
            normalize(text)
        )
    )


def is_decision(text):

    n = normalize(text)

    return (
        re.match(
            r"^قرار\s*مجلس\s*النواب\b",
            n
        )
        or
        re.match(
            r"^قرارمجلس\s*النواب\b",
            n
        )
        or
        re.match(
            r"^قرار\s+مجلس\b",
            n
        )
    )


def major_type(text):

    if is_constitutional(text):
        return "constitutional_amendment"

    if is_law(text):
        return "law"

    if is_decision(text):
        return "decision"

    return None


# ============================================================
# ARTICLE DETECTOR
# ============================================================

def is_article(text):

    n = normalize(text)

    return bool(
        re.match(
            r"^الماده\b",
            n
        )
        or
        re.match(
            r"^المادة\b",
            n
        )
    )


# ============================================================
# PREAMBLE
# ============================================================

def is_preamble(text):

    n = normalize(text)

    patterns = [
        r"^مجلس\s+النواب",
        r"^بعد\s+الاطلاع",
        r"^وعلي",
        r"^وعلى",
        r"^باسم",
        r"^بناء\s+على",
        r"^استنادا",
        r"^استناداً"
    ]

    return any(
        re.match(
            p,
            n
        )
        for p in patterns
    )


# ============================================================
# BUILD LEGAL SECTIONS
#
# We only use the beginning of OCR elements to detect a new
# legal document. References to laws inside a paragraph won't
# accidentally become new sections.
# ============================================================

sections = []

current = None

for unit in units:

    page = unit["page"]
    text = unit["text"].strip()

    # Pages 1-2 = table of contents
    if page in [1, 2]:
        continue

    detected = major_type(text)

    if detected:

        if current is not None:
            sections.append(
                current
            )

        current = {
            "type": detected,
            "title": text,
            "page_start": page,
            "page_end": page,
            "units": []
        }

        continue

    if current is None:
        continue

    current["units"].append(
        unit
    )

    current["page_end"] = page


if current is not None:
    sections.append(
        current
    )


# ============================================================
# DOCUMENT-WIDE DF
# ============================================================

df = Counter()

for unit in units:

    if unit["page"] in [1, 2]:
        continue

    unique = set(
        words(unit["text"])
    )

    for word in unique:

        if (
            len(word) >= 3
            and word not in STOPWORDS
            and word not in GENERIC
        ):
            df[word] += 1


TOTAL_UNITS = len(units)


# ============================================================
# KEYWORDS
# ============================================================

def get_keywords(text, n=10):

    counter = Counter(
        w for w in words(text)
        if (
            len(w) >= 3
            and w not in STOPWORDS
            and w not in GENERIC
        )
    )

    scored = []

    for word, tf in counter.items():

        word_df = df.get(
            word,
            1
        )

        idf = math.log(
            (TOTAL_UNITS + 1)
            /
            (word_df + 1)
        ) + 1

        score = tf * idf

        scored.append(
            (
                word,
                score
            )
        )

    scored.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        word
        for word, score in scored[:n]
    ]


# ============================================================
# KEYWORD SIMILARITY
# ============================================================

def similarity(a, b):

    if not a or not b:
        return 0

    return len(
        a & b
    ) / len(
        a | b
    )


# ============================================================
# IDENTIFY NUMBERED CLAUSES
# ============================================================

def numbered_clause(text):

    n = normalize(text)

    patterns = [
        r"^[0-9]+\s+",
        r"^[٠-٩]+\s+",
        r"^[0-9]+[-.)]",
        r"^[٠-٩]+[-.)]",
        r"^[ابتثجحخدذرزسشصضطظعغفقكلمنهوي]-"
    ]

    return any(
        re.match(
            p,
            n
        )
        for p in patterns
    )


# ============================================================
# SPLIT ARTICLE INTO MEANINGFUL TOPICS
#
# IMPORTANT:
# We do NOT split every time keywords differ.
#
# A topic split requires:
#   - enough OCR blocks already accumulated
#   - meaningful keyword difference
#   - the incoming block is substantial
#
# This prevents 5-word chunks.
# ============================================================

def split_topics(article_units):

    if not article_units:
        return []


    # --------------------------------------------------------
    # First build natural groups from OCR elements.
    #
    # We use topic changes only when there is strong evidence.
    # --------------------------------------------------------

    groups = []

    current = []

    current_keywords = set()

    for unit in article_units:

        text = unit["text"]

        incoming = set(
            get_keywords(
                text,
                10
            )
        )

        if not current:

            current.append(
                unit
            )

            current_keywords.update(
                incoming
            )

            continue


        # Numbered clauses are useful boundaries,
        # but only when we already have a meaningful group.
        clause = numbered_clause(
            text
        )

        # Count actual words already accumulated
        current_word_count = sum(
            len(
                x["text"].split()
            )
            for x in current
        )

        sim = similarity(
            current_keywords,
            incoming
        )

        # Strong topic change
        strong_topic_change = (
            len(current) >= 4
            and current_word_count >= 35
            and len(incoming) >= 3
            and sim < 0.10
        )

        # Moderate topic change
        moderate_topic_change = (
            len(current) >= 6
            and current_word_count >= 50
            and len(incoming) >= 4
            and sim < 0.16
        )

        should_split = (
            clause
            and current_word_count >= 45
        ) or strong_topic_change or moderate_topic_change


        if should_split:

            groups.append(
                current
            )

            current = [
                unit
            ]

            current_keywords = set(
                incoming
            )

        else:

            current.append(
                unit
            )

            current_keywords.update(
                incoming
            )


    if current:

        groups.append(
            current
        )


    return groups


# ============================================================
# CREATE CHUNKS
# ============================================================

chunks = []

for section_number, section in enumerate(
    sections,
    start=1
):

    title = section["title"]

    body = section["units"]

    # --------------------------------------------------------
    # Locate articles
    # --------------------------------------------------------

    articles = []

    current_article = None
    current_body = []

    for unit in body:

        text = unit["text"]

        if is_article(text):

            if current_article is not None:

                articles.append({
                    "title": current_article,
                    "body": current_body
                })

            current_article = text

            current_body = []

        else:

            current_body.append(
                unit
            )


    if current_article is not None:

        articles.append({
            "title": current_article,
            "body": current_body
        })


    # --------------------------------------------------------
    # NO ARTICLES
    # --------------------------------------------------------

    if not articles:

        groups = split_topics(
            body
        )

        for group in groups:

            text = " ".join(
                x["text"]
                for x in group
            )

            if not text.strip():
                continue

            chunks.append({

                "chunk_id":
                    f"CH{len(chunks)+1:04d}",

                "section_id":
                    f"SEC{section_number:03d}",

                "section_title":
                    title,

                "section_type":
                    section["type"],

                "article":
                    None,

                "chunk_kind":
                    "section_topic",

                "pages":
                    sorted(
                        set(
                            x["page"]
                            for x in group
                        )
                    ),

                "keywords":
                    get_keywords(
                        text,
                        12
                    ),

                "text":
                    text.strip(),

                "word_count":
                    len(
                        text.split()
                    ),

                "source":
                    os.path.basename(
                        INPUT_FILE
                    ),

                "include_in_embeddings":
                    True
            })

        continue


    # --------------------------------------------------------
    # ARTICLES
    # --------------------------------------------------------

    for article_index, article in enumerate(
        articles,
        start=1
    ):

        article_title = article[
            "title"
        ]

        article_body = article[
            "body"
        ]


        # ----------------------------------------------------
        # EVERYTHING BEFORE FIRST ARTICLE
        #
        # This fixes the 820-word coverage problem.
        # ----------------------------------------------------

        if article_index == 1:

            # The section preamble is not inside the article.
            #
            # We intentionally preserve it as its own chunk.
            #
            # If article_body contains only article content,
            # no separate preamble exists.
            pass


        # ----------------------------------------------------
        # Split article
        # ----------------------------------------------------

        topic_groups = split_topics(
            article_body
        )


        # ----------------------------------------------------
        # If article has no body
        # ----------------------------------------------------

        if not topic_groups:

            topic_groups = [
                []
            ]


        # ----------------------------------------------------
        # Create chunks
        # ----------------------------------------------------

        for topic_index, group in enumerate(
            topic_groups,
            start=1
        ):

            text_parts = []

            # Article heading always travels with its content.
            text_parts.append(
                article_title
            )

            for unit in group:

                text_parts.append(
                    unit["text"]
                )

            text = " ".join(
                x.strip()
                for x in text_parts
                if x.strip()
            )

            if not text.strip():
                continue

            pages_used = sorted(
                set(
                    x["page"]
                    for x in group
                )
            )

            # If article heading isn't represented by body,
            # add its nearest page.
            if not pages_used:

                pages_used = [
                    section["page_start"]
                ]


            if len(topic_groups) == 1:

                kind = "article"

            else:

                kind = "article_topic"


            chunks.append({

                "chunk_id":
                    f"CH{len(chunks)+1:04d}",

                "section_id":
                    f"SEC{section_number:03d}",

                "section_title":
                    title,

                "section_type":
                    section["type"],

                "article":
                    article_title,

                "article_number":
                    article_index,

                "topic_number":
                    topic_index,

                "chunk_kind":
                    kind,

                "pages":
                    pages_used,

                "keywords":
                    get_keywords(
                        text,
                        12
                    ),

                "text":
                    text.strip(),

                "word_count":
                    len(
                        text.split()
                    ),

                "source":
                    os.path.basename(
                        INPUT_FILE
                    ),

                "include_in_embeddings":
                    True
            })


# ============================================================
# IMPORTANT:
# ADD SECTION PREAMBLES
#
# We need to recover text before the first "المادة".
# ============================================================

preamble_chunks = []

for section_number, section in enumerate(
    sections,
    start=1
):

    body = section["units"]

    # Find first article
    first_article_index = None

    for i, unit in enumerate(body):

        if is_article(
            unit["text"]
        ):

            first_article_index = i
            break


    if first_article_index is None:
        continue


    preamble = body[
        :first_article_index
    ]


    if not preamble:
        continue


    text = " ".join(
        x["text"]
        for x in preamble
    ).strip()


    if not text:
        continue


    preamble_chunks.append({

        "section_number":
            section_number,

        "section_id":
            f"SEC{section_number:03d}",

        "section_title":
            section["title"],

        "section_type":
            section["type"],

        "pages":
            sorted(
                set(
                    x["page"]
                    for x in preamble
                )
            ),

        "text":
            text,

        "word_count":
            len(
                text.split()
            ),

        "keywords":
            get_keywords(
                text,
                12
            )
    })


# ============================================================
# INSERT PREAMBLES BEFORE FIRST ARTICLE OF EACH SECTION
# ============================================================

final_chunks = []

for section_number, section in enumerate(
    sections,
    start=1
):

    # --------------------------------------------------------
    # Preamble
    # --------------------------------------------------------

    matching_preambles = [
        x
        for x in preamble_chunks
        if x["section_number"]
        == section_number
    ]

    for preamble in matching_preambles:

        final_chunks.append({

            "chunk_id":
                "",

            "section_id":
                preamble["section_id"],

            "section_title":
                preamble["section_title"],

            "section_type":
                preamble["section_type"],

            "article":
                None,

            "article_number":
                None,

            "topic_number":
                None,

            "chunk_kind":
                "preamble",

            "pages":
                preamble["pages"],

            "keywords":
                preamble["keywords"],

            "text":
                preamble["text"],

            "word_count":
                preamble["word_count"],

            "source":
                os.path.basename(
                    INPUT_FILE
                ),

            "include_in_embeddings":
                True
        })


    # --------------------------------------------------------
    # Existing chunks belonging to section
    # --------------------------------------------------------

    for chunk in chunks:

        if chunk["section_id"] == (
            f"SEC{section_number:03d}"
        ):

            final_chunks.append(
                chunk
            )


# ============================================================
# RENUMBER CHUNKS
# ============================================================

for index, chunk in enumerate(
    final_chunks,
    start=1
):

    chunk["chunk_id"] = (
        f"CH{index:04d}"
    )


# ============================================================
# TABLE OF CONTENTS
# ============================================================

contents = []

for page in pages:

    if page.get("page") not in [1, 2]:
        continue

    text = page.get(
        "text",
        ""
    ).strip()

    if text:
        contents.append(
            text
        )


contents_text = "\n".join(
    contents
).strip()


if contents_text:

    contents_chunk = {

        "chunk_id":
            "CH0000",

        "section_id":
            "CONTENTS",

        "section_title":
            "محتويات العدد",

        "section_type":
            "table_of_contents",

        "article":
            None,

        "article_number":
            None,

        "topic_number":
            None,

        "chunk_kind":
            "contents",

        "pages":
            [1, 2],

        "keywords":
            get_keywords(
                contents_text,
                15
            ),

        "text":
            contents_text,

        "word_count":
            len(
                contents_text.split()
            ),

        "source":
            os.path.basename(
                INPUT_FILE
            ),

        "include_in_embeddings":
            False
    }

    final_chunks.insert(
        0,
        contents_chunk
    )


# ============================================================
# COVERAGE CHECK
# ============================================================

source_words_all = 0

for page in pages:

    text = page.get(
        "text",
        ""
    ).strip()

    source_words_all += len(
        text.split()
    )


source_words_legal = 0

for page in pages:

    if page.get("page") < 3:
        continue

    text = page.get(
        "text",
        ""
    ).strip()

    source_words_legal += len(
        text.split()
    )


chunk_words_legal = sum(
    chunk["word_count"]
    for chunk in final_chunks
    if chunk["include_in_embeddings"]
)


coverage_difference = (
    source_words_legal
    -
    chunk_words_legal
)


# ============================================================
# SAVE
# ============================================================

output = {

    "source_pdf":
        data.get(
            "source_pdf"
        ),

    "source_ocr":
        INPUT_FILE,

    "chunking_version":
        "legal_rag_v3",

    "method": [

        "legal section detection",

        "law detection",

        "decision detection",

        "constitutional amendment detection",

        "article detection",

        "legal preamble preservation",

        "keyword-based topic detection",

        "numbered clause detection",

        "semantic grouping",

        "full source coverage verification"
    ],

    "pages":
        len(pages),

    "pages_with_text":
        sum(
            bool(
                p.get(
                    "text",
                    ""
                ).strip()
            )
            for p in pages
        ),

    "source_words_all":
        source_words_all,

    "source_words_legal":
        source_words_legal,

    "total_chunks":
        len(final_chunks),

    "embedding_chunks":
        sum(
            c["include_in_embeddings"]
            for c in final_chunks
        ),

    "chunk_words":
        chunk_words_legal,

    "coverage_difference":
        coverage_difference,

    "coverage_ok":
        coverage_difference == 0,

    "chunks":
        final_chunks
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# PREVIEW
# ============================================================

with open(
    PREVIEW_FILE,
    "w",
    encoding="utf-8"
) as f:

    for chunk in final_chunks:

        f.write(
            "\n"
            + "=" * 75
            + "\n"
        )

        f.write(
            f"{chunk['chunk_id']}\n"
        )

        f.write(
            f"Type: {chunk['chunk_kind']}\n"
        )

        f.write(
            f"Section: {chunk['section_title']}\n"
        )

        f.write(
            f"Article: {chunk.get('article')}\n"
        )

        f.write(
            f"Pages: {chunk['pages']}\n"
        )

        f.write(
            "Keywords: "
            + ", ".join(
                chunk["keywords"]
            )
            + "\n"
        )

        f.write(
            "-" * 75
            + "\n"
        )

        f.write(
            chunk["text"]
        )

        f.write("\n")


# ============================================================
# FINAL REPORT
# ============================================================

print()
print("=" * 75)
print("FINAL CHUNKING REPORT")
print("=" * 75)

print(
    "Source words (all pages):",
    source_words_all
)

print(
    "Legal words (pages 3+):",
    source_words_legal
)

print(
    "Total chunks:",
    len(final_chunks)
)

print(
    "Embedding chunks:",
    sum(
        c["include_in_embeddings"]
        for c in final_chunks
    )
)

print(
    "Words represented:",
    chunk_words_legal
)

print(
    "Coverage difference:",
    coverage_difference
)

print(
    "Coverage OK:",
    coverage_difference == 0
)

print()
print(
    "JSON:",
    OUTPUT_FILE
)

print(
    "Preview:",
    PREVIEW_FILE
)


# ============================================================
# CHUNK SUMMARY
# ============================================================

print()
print("=" * 75)
print("CHUNK SUMMARY")
print("=" * 75)

for chunk in final_chunks:

    print(
        f"{chunk['chunk_id']} | "
        f"{chunk['chunk_kind']:14s} | "
        f"{chunk['word_count']:4d} words | "
        f"Pages {chunk['pages']}"
    )

    print(
        f"   {chunk['section_title']}"
    )

    if chunk.get("article"):
        print(
            f"   {chunk['article']}"
        )

    print(
        "   Keywords:",
        ", ".join(
            chunk["keywords"][:8]
        )
    )

In [ ]:
import os
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# ============================================================
# STEP 4 — GENERATE EMBEDDINGS
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm\day09_rag"

CHUNKS_FILE = os.path.join(
    BASE_DIR, "03_chunks", "chunks.json"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR, "04_embeddings"
)

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR, "embeddings.json"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load chunks
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

chunks = data["chunks"]

print(f"Loaded chunks: {len(chunks)}")

# Load multilingual embedding model
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(MODEL_NAME)

# Prepare embedding text
texts = []

for chunk in chunks:

    section = chunk.get("section_title", "")
    article = chunk.get("article_title", "")
    text = chunk.get("text", "")

    embedding_text = "\n".join(
        x for x in [section, article, text] if x
    )

    texts.append(embedding_text)

# Generate embeddings
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Embedding shape: {embeddings.shape}")

# Save
output = []

for chunk, embedding in zip(chunks, embeddings):

    output.append({
        "chunk_id": chunk["chunk_id"],
        "embedding": embedding.tolist()
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        output,
        f,
        ensure_ascii=False
    )

print()
print("Embeddings saved successfully:")
print(OUTPUT_FILE)

In [ ]:
import os
import json
import numpy as np
import faiss

# ============================================================
# STEP 5 — BUILD FAISS VECTOR STORE
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm\day09_rag"

EMBEDDINGS_FILE = os.path.join(
    BASE_DIR, "04_embeddings", "embeddings.json"
)

CHUNKS_FILE = os.path.join(
    BASE_DIR, "03_chunks", "chunks.json"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR, "05_vector_store"
)

INDEX_FILE = os.path.join(
    OUTPUT_DIR, "index.faiss"
)

METADATA_FILE = os.path.join(
    OUTPUT_DIR, "metadata.json"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load embeddings
with open(EMBEDDINGS_FILE, "r", encoding="utf-8") as f:
    embedding_data = json.load(f)

# Load chunks
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunk_data = json.load(f)

chunks = chunk_data["chunks"]

# Convert embeddings to NumPy
vectors = np.array(
    [item["embedding"] for item in embedding_data],
    dtype="float32"
)

print("Vectors:", vectors.shape)
print("Chunks:", len(chunks))

# Create FAISS index
dimension = vectors.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(vectors)

print("FAISS vectors:", index.ntotal)

# Save FAISS index
faiss.write_index(index, INDEX_FILE)

# Save metadata
metadata = []

for chunk in chunks:

    metadata.append({
        "chunk_id": chunk["chunk_id"],
        "chunk_kind": chunk.get("chunk_kind", ""),
        "pages": chunk.get("pages", []),
        "section_title": chunk.get("section_title", ""),
        "article_title": chunk.get("article_title", ""),
        "text": chunk.get("text", "")
    })

with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print()
print("FAISS index saved:")
print(INDEX_FILE)

print()
print("Metadata saved:")
print(METADATA_FILE)

In [ ]:
import os
import json
import faiss
from sentence_transformers import SentenceTransformer

# ============================================================
# STEP 6 — TEST SEMANTIC SEARCH
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm\day09_rag"

INDEX_FILE = os.path.join(
    BASE_DIR, "05_vector_store", "index.faiss"
)

METADATA_FILE = os.path.join(
    BASE_DIR, "05_vector_store", "metadata.json"
)

# Load embedding model
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Load FAISS
index = faiss.read_index(INDEX_FILE)

# Load metadata
with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("FAISS vectors:", index.ntotal)
print("Metadata:", len(metadata))


def search(query, top_k=5):

    query_vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_vector,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        item = metadata[idx]

        results.append({
            "score": float(score),
            "chunk_id": item["chunk_id"],
            "pages": item["pages"],
            "section": item["section_title"],
            "article": item["article_title"],
            "text": item["text"]
        })

    return results


# ============================================================
# TEST QUESTIONS
# ============================================================

questions = [
    "ما هو التعديل الدستوري الثاني عشر؟",
    "من تم تكليفه برئاسة الحكومة الليبية؟",
    "ما هو قرار سحب الثقة من حكومة الوحدة الوطنية؟",
    "من تم تكليفه برئاسة المحكمة العليا؟",
    "ما هو القرار المتعلق بنائب محافظ مصرف ليبيا المركزي؟"
]

for question in questions:

    print("\n")
    print("=" * 100)
    print("QUESTION:")
    print(question)
    print("=" * 100)

    results = search(question, top_k=3)

    for i, result in enumerate(results, 1):

        print(f"\nRESULT {i}")
        print("-" * 80)
        print("Chunk:", result["chunk_id"])
        print("Score:", round(result["score"], 4))
        print("Pages:", result["pages"])

        if result["section"]:
            print("Section:", result["section"])

        if result["article"]:
            print("Article:", result["article"])

        print()
        print(result["text"])

In [ ]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = r"C:\Users\PC-LAB1\Desktop\llm\day09_rag"

EMBEDDINGS_FILE = os.path.join(
    PROJECT_DIR, "04_embeddings", "embeddings.json"
)

INDEX_FILE = os.path.join(
    PROJECT_DIR, "05_vector_store", "index.faiss"
)

METADATA_FILE = os.path.join(
    PROJECT_DIR, "05_vector_store", "metadata.json"
)

RESULTS_DIR = os.path.join(
    PROJECT_DIR, "06_results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)


# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Loading embedding model...")
model = SentenceTransformer(MODEL_NAME)


# ============================================================
# LOAD FAISS INDEX
# ============================================================

print("Loading FAISS index...")
index = faiss.read_index(INDEX_FILE)


# ============================================================
# LOAD METADATA
# ============================================================

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"FAISS vectors : {index.ntotal}")
print(f"Metadata items: {len(metadata)}")


# ============================================================
# RAG RETRIEVAL FUNCTION
# ============================================================

def rag_retrieve(query, top_k=5):

    # --------------------------------------------------------
    # 1. User Arabic Query
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("USER QUERY")
    print("=" * 70)
    print(query)

    # --------------------------------------------------------
    # 2. Query Embedding
    # --------------------------------------------------------

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    # --------------------------------------------------------
    # 3. Semantic Search
    # --------------------------------------------------------

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    # --------------------------------------------------------
    # 4. Top-K Chunks
    # --------------------------------------------------------

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        if idx < 0:
            continue

        item = metadata[idx]

        result = {
            "rank": rank,
            "score": float(score),
            "chunk_id": item.get("chunk_id"),
            "chunk_kind": item.get("chunk_kind"),
            "pages": item.get("pages"),
            "section_title": item.get("section_title"),
            "article_title": item.get("article_title"),
            "text": item.get("text"),

            # Source document
            "document": "الجريدة الرسمية - دولة ليبيا"
        }

        results.append(result)

    # --------------------------------------------------------
    # 5. Documents + URLs
    # --------------------------------------------------------

    output = {
        "query": query,
        "top_k": top_k,
        "results": results
    }

    return output


# ============================================================
# TEST QUERY
# ============================================================

query = "من تم تكليفه برئاسة الحكومة الليبية؟"

rag_result = rag_retrieve(
    query,
    top_k=5
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("TOP-K RETRIEVED DOCUMENTS")
print("=" * 70)

for result in rag_result["results"]:

    print(f"\nRank : {result['rank']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Chunk: {result['chunk_id']}")
    print(f"Pages: {result['pages']}")

    if result["section_title"]:
        print(f"Section: {result['section_title']}")

    if result["article_title"]:
        print(f"Article: {result['article_title']}")

    print(f"Document: {result['document']}")

    print("\nText:")
    print(result["text"])


# ============================================================
# SAVE RESULTS
# ============================================================

RESULT_FILE = os.path.join(
    RESULTS_DIR,
    "rag_retrieval_result.json"
)

with open(
    RESULT_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        rag_result,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n" + "=" * 70)
print("RAG RETRIEVAL COMPLETE")
print("=" * 70)
print(f"Results saved to:")
print(RESULT_FILE)

In [ ]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_DIR = r"C:\Users\PC-LAB1\Desktop\llm\day09_rag"

INDEX_FILE = os.path.join(
    PROJECT_DIR, "05_vector_store", "index.faiss"
)

METADATA_FILE = os.path.join(
    PROJECT_DIR, "05_vector_store", "metadata.json"
)

RESULTS_DIR = os.path.join(
    PROJECT_DIR, "06_results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Number of results to retrieve
TOP_K = 5


# ============================================================
# LOAD MODEL
# ============================================================

print("Loading embedding model...")

model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded.")


# ============================================================
# LOAD FAISS
# ============================================================

print("Loading FAISS vector store...")

index = faiss.read_index(INDEX_FILE)

print(f"FAISS vectors: {index.ntotal}")


# ============================================================
# LOAD METADATA
# ============================================================

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Metadata items: {len(metadata)}")


# ============================================================
# RAG RETRIEVAL
# ============================================================

def retrieve(query, top_k=TOP_K):

    # --------------------------------------------------------
    # 1. User Arabic Query
    # --------------------------------------------------------

    # --------------------------------------------------------
    # 2. Query Embedding
    # --------------------------------------------------------

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    # --------------------------------------------------------
    # 3. Semantic Search
    # --------------------------------------------------------

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    # --------------------------------------------------------
    # 4. Top-K Chunks
    # --------------------------------------------------------

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        if idx < 0 or idx >= len(metadata):
            continue

        item = metadata[idx]

        result = {
            "rank": rank,
            "score": float(score),
            "chunk_id": item.get("chunk_id"),
            "chunk_kind": item.get("chunk_kind"),
            "pages": item.get("pages"),
            "section_title": item.get("section_title"),
            "article_title": item.get("article_title"),
            "text": item.get("text"),
            "document": "الجريدة الرسمية - دولة ليبيا",
            "source": "الجريدة-الرسمية-دولة-ليبيا-ديوان-مجلس-النواب-العدد-الثاني.pdf"
        }

        results.append(result)

    return results


# ============================================================
# DISPLAY FUNCTION
# ============================================================

def display_results(query, results):

    print("\n")
    print("=" * 80)
    print("RAG RETRIEVAL RESULT")
    print("=" * 80)

    print("\nUser Arabic Query:")
    print(query)

    print("\n" + "-" * 80)
    print(f"Top-{len(results)} Retrieved Chunks")
    print("-" * 80)

    for result in results:

        print("\n" + "=" * 80)

        print(f"Rank       : {result['rank']}")
        print(f"Similarity : {result['score']:.4f}")
        print(f"Chunk ID   : {result['chunk_id']}")
        print(f"Pages      : {result['pages']}")

        if result["section_title"]:
            print(f"Section    : {result['section_title']}")

        if result["article_title"]:
            print(f"Article    : {result['article_title']}")

        print(f"Document   : {result['document']}")

        print("\nRetrieved Document Text:")
        print(result["text"])

    print("\n" + "=" * 80)


# ============================================================
# INTERACTIVE RAG LOOP
# ============================================================

print("\n")
print("=" * 80)
print("ARABIC RAG SEMANTIC SEARCH")
print("=" * 80)

print("\nEnter an Arabic question.")
print("Type 'exit' to stop.")

while True:

    query = input("\nالسؤال: ").strip()

    if query.lower() in ["exit", "quit", "q", "خروج"]:
        print("\nRAG search stopped.")
        break

    if not query:
        print("Please enter a question.")
        continue

    # Retrieve Top-K chunks
    results = retrieve(
        query,
        top_k=TOP_K
    )

    # Display results
    display_results(
        query,
        results
    )

    # --------------------------------------------------------
    # Save current result
    # --------------------------------------------------------

    output = {
        "query": query,
        "top_k": TOP_K,
        "results": results
    }

    result_file = os.path.join(
        RESULTS_DIR,
        "rag_retrieval_result.json"
    )

    with open(
        result_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            output,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"\nResult saved to:")
    print(result_file)